# Module 2 — LLM Foundations for AI Engineers

**Hands-on objective:** Make core LLM concepts visible through experiments — tokenisation, context windows, embeddings, reasoning behaviour, hallucination, model limitations, model comparison, and prompt effectiveness.



## Tokenisation basics

**What this demonstrates:** Convert human-readable text into the token IDs that the model actually processes.

**What to observe:** Word count and token count differ because tokenizers split text into model-specific subword units.


In [2]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "Prior authorization is required for MRI procedures."

tokens = encoding.encode(text)

print("Original text:", text)
print("Word count:", len(text.split()))
print("Token count:", len(tokens))
print("Token IDs:", tokens)

Original text: Prior authorization is required for MRI procedures.
Word count: 7
Token count: 8
Token IDs: [50571, 24645, 374, 2631, 369, 52460, 16346, 13]


**Takeaway:** LLMs do not read words the way humans do — they operate on token sequences, and those tokens drive context limits and cost.


## Secure configuration loading

**What this demonstrates:** Load endpoint, credential, and deployment settings from the `.env` file instead of hard-coding secrets in the notebook.

**What to observe:** The notebook should confirm that configuration values exist without exposing the API key itself.


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure model client ready")

Azure model client ready


**Takeaway:** The same code can now move across environments by changing configuration, not source code — a small design choice that scales into production.


## Inspect token boundaries

**What this demonstrates:** Decode individual token IDs to see how words, spaces, and punctuation are represented.

**What to observe:** Notice that tokens can include leading spaces or partial text segments rather than mapping one-to-one with words.


In [4]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} -> {repr(token_text)}")

50571 -> 'Prior'
24645 -> ' authorization'
374 -> ' is'
2631 -> ' required'
369 -> ' for'
52460 -> ' MRI'
16346 -> ' procedures'
13 -> '.'


**Takeaway:** A visually short string can be computationally expensive if it tokenizes inefficiently.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [5]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    token_ids = encoding.encode(text)

    print("\nTEXT:", text)
    print("Words :", len(text.split()))
    print("Tokens:", len(token_ids))


TEXT: Prior authorization is required for MRI procedures.
Words : 7
Tokens: 8

TEXT: PA is required for MRI.
Words : 5
Tokens: 6

TEXT: The member's deductible is $1,500.
Words : 5
Tokens: 10

TEXT: AuthorizationRequired=True
Words : 1
Tokens: 3

TEXT: पूर्व प्राधिकरण आवश्यक है
Words : 4
Tokens: 27


**Takeaway:** Every notebook cell should answer one engineering question: what changed, why did it change, and why does it matter in production?


## Token efficiency comparison

**What this demonstrates:** Compare how different text styles and languages consume tokens per visible word.

**What to observe:** Numbers, symbols, code-like strings, and multilingual text can tokenize very differently from ordinary English prose.


In [6]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    words = len(text.split())
    tokens = len(encoding.encode(text))
    ratio = tokens / words if words > 0 else 0

    print(f"\n{text}")
    print(f"Words            : {words}")
    print(f"Tokens           : {tokens}")
    print(f"Tokens per word  : {ratio:.2f}")


Prior authorization is required for MRI procedures.
Words            : 7
Tokens           : 8
Tokens per word  : 1.14

PA is required for MRI.
Words            : 5
Tokens           : 6
Tokens per word  : 1.20

The member's deductible is $1,500.
Words            : 5
Tokens           : 10
Tokens per word  : 2.00

AuthorizationRequired=True
Words            : 1
Tokens           : 3
Tokens per word  : 3.00

पूर्व प्राधिकरण आवश्यक है
Words            : 4
Tokens           : 27
Tokens per word  : 6.75


**Takeaway:** Global AI systems can have very different token economics across languages even when users type the same number of words.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [7]:
prompt_text = "Explain prior authorization in healthcare in two sentences."

local_token_count = len(encoding.encode(prompt_text))

api_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": prompt_text
        }
    ]
)

print("Local text token estimate :", local_token_count)
print("Azure prompt tokens       :", api_response.usage.prompt_tokens)

Local text token estimate : 10
Azure prompt tokens       : 16


**Takeaway:** This cell converts a local notebook into a cloud-connected AI application.


## Context-window setup

**What this demonstrates:** Create short and longer policy contexts for the same question so we can isolate the effect of context size.

**What to observe:** Only the amount of supplied context changes; the question stays constant.


In [8]:
short_context = """
Policy: MRI procedures require prior authorization.
"""

long_context = """
Policy: MRI procedures require prior authorization.
CT scans require prior authorization only for outpatient procedures.
Emergency room imaging does not require prior authorization.
Physical therapy requires authorization after 10 visits.
Specialist consultations do not require prior authorization.
"""

question = "Does an MRI require prior authorization?"

**Takeaway:** This is the foundation of a key RAG principle: the model does not need all available information — it needs the right information.


## Short-context baseline

**What this demonstrates:** Answer the policy question using only the minimum relevant context.

**What to observe:** This gives us a baseline for both answer quality and prompt-token consumption.


In [11]:
short_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{short_context}

Question:
{question}
"""
        }
    ]
)

print(short_response.choices[0].message.content)
print("\nPrompt tokens:", short_response.usage.prompt_tokens)

Yes, an MRI requires prior authorization according to the policy.

Prompt tokens: 39


**Takeaway:** A compact, relevant context often gives the model everything it needs.


## Longer-context comparison

**What this demonstrates:** Ask the same question with more policy information in the prompt.

**What to observe:** The answer may remain unchanged while prompt-token usage increases.


In [9]:
long_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{long_context}

Question:
{question}
"""
        }
    ]
)

print(long_response.choices[0].message.content)
print("\nPrompt tokens:", long_response.usage.prompt_tokens)

Yes, an MRI procedure requires prior authorization according to the policy.

Prompt tokens: 76


**Takeaway:** More context is not automatically more intelligence — sometimes it is simply more cost.


## Short-context baseline

**What this demonstrates:** Answer the policy question using only the minimum relevant context.

**What to observe:** This gives us a baseline for both answer quality and prompt-token consumption.


In [12]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

Short-context tokens : 39
Long-context tokens  : 76
Additional tokens    : 37
Increase             : 94.9%


**Takeaway:** A compact, relevant context often gives the model everything it needs.


## Short-context baseline

**What this demonstrates:** Answer the policy question using only the minimum relevant context.

**What to observe:** This gives us a baseline for both answer quality and prompt-token consumption.


In [13]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

Short-context tokens : 39
Long-context tokens  : 76
Additional tokens    : 37
Increase             : 94.9%


**Takeaway:** A compact, relevant context often gives the model everything it needs.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [14]:
noisy_context = """
Member ID: M102938
Plan Type: PPO
Primary Care Copay: $25
Specialist Copay: $50
Emergency Room Copay: $250
Dental coverage is not included.
Vision coverage is included once every 24 months.
Physical therapy requires authorization after 10 visits.
MRI procedures require prior authorization.
Member mailing address was updated last month.
Claims are processed within standard turnaround time.
"""

question = "Does an MRI require prior authorization?"

**Takeaway:** Every notebook cell should answer one engineering question: what changed, why did it change, and why does it matter in production?


## Noisy-context experiment

**What this demonstrates:** Mix the relevant MRI rule with unrelated member and benefit information and observe the impact.

**What to observe:** The model can still find the answer, but the prompt consumes more tokens and carries more distracting information.


In [15]:
noisy_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{noisy_context}

Question:
{question}
"""
        }
    ]
)

print(noisy_response.choices[0].message.content)
print("\nPrompt tokens:", noisy_response.usage.prompt_tokens)

Yes, an MRI procedure requires prior authorization.

Prompt tokens: 114


**Takeaway:** Enterprise RAG quality depends as much on what you exclude as on what you retrieve.


## Short-context baseline

**What this demonstrates:** Answer the policy question using only the minimum relevant context.

**What to observe:** This gives us a baseline for both answer quality and prompt-token consumption.


In [16]:
print("Short context tokens :", short_response.usage.prompt_tokens)
print("Long context tokens  :", long_response.usage.prompt_tokens)
print("Noisy context tokens :", noisy_response.usage.prompt_tokens)

Short context tokens : 39
Long context tokens  : 76
Noisy context tokens : 114


**Takeaway:** A compact, relevant context often gives the model everything it needs.


## Short-context baseline

**What this demonstrates:** Answer the policy question using only the minimum relevant context.

**What to observe:** This gives us a baseline for both answer quality and prompt-token consumption.


In [17]:
import pandas as pd

context_comparison = pd.DataFrame([
    {
        "Scenario": "Short context",
        "Prompt Tokens": short_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "High"
    },
    {
        "Scenario": "Long context",
        "Prompt Tokens": long_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Medium"
    },
    {
        "Scenario": "Noisy context",
        "Prompt Tokens": noisy_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Low"
    }
])

context_comparison

,Scenario,Prompt Tokens,Answer Quality,Context Relevance
0,Short context,39,Correct,High
1,Long context,76,Correct,Medium
2,Noisy context,114,Correct,Low


**Takeaway:** A compact, relevant context often gives the model everything it needs.


## Embedding scenario setup

**What this demonstrates:** Prepare semantically related and unrelated healthcare payer sentences for vector comparison.

**What to observe:** The samples are designed so similar meanings use different wording.


In [18]:
embedding_samples = [
    "Prior authorization is required for MRI procedures.",
    "MRI scans need approval from the health insurance payer.",
    "The member updated their mailing address.",
    "The deductible must be paid before the health plan starts sharing costs."
]

for i, text in enumerate(embedding_samples, start=1):
    print(f"{i}. {text}")

1. Prior authorization is required for MRI procedures.
2. MRI scans need approval from the health insurance payer.
3. The member updated their mailing address.
4. The deductible must be paid before the health plan starts sharing costs.


**Takeaway:** Embeddings let systems compare meaning rather than exact keywords.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [19]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

Embedding model configured: True
Embedding deployment       : text-embedding-3-small


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [20]:
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

Embedding model configured: True
Embedding deployment       : text-embedding-3-small


## Generate text embeddings

**What this demonstrates:** Convert each healthcare sentence into a high-dimensional numeric vector using the Azure embedding model.

**What to observe:** Each text becomes a fixed-length vector that can be compared mathematically.


In [21]:
embedding_response = client.embeddings.create(
    model=embedding_model,
    input=embedding_samples
)

embeddings = [item.embedding for item in embedding_response.data]

print("Number of embeddings:", len(embeddings))
print("Embedding dimension :", len(embeddings[0]))

Number of embeddings: 4
Embedding dimension : 1536


**Trainer takeaway:** Once language becomes vectors, semantic search becomes a geometry problem.


## Embedding scenario setup

**What this demonstrates:** Prepare semantically related and unrelated healthcare payer sentences for vector comparison.

**What to observe:** The samples are designed so similar meanings use different wording.


In [22]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

for i in range(len(embedding_samples)):
    for j in range(i + 1, len(embedding_samples)):
        score = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

        print(f"{i+1} vs {j+1}: {score:.4f}")

1 vs 2: 0.6702
1 vs 3: 0.0765
1 vs 4: 0.3587
2 vs 3: 0.1100
2 vs 4: 0.4466
3 vs 4: 0.0513


**Takeaway:** Embeddings let systems compare meaning rather than exact keywords.


## Embedding scenario setup

**What this demonstrates:** Prepare semantically related and unrelated healthcare payer sentences for vector comparison.

**What to observe:** The samples are designed so similar meanings use different wording.


In [23]:
import pandas as pd
import numpy as np

similarity_matrix = np.zeros((len(embedding_samples), len(embedding_samples)))

for i in range(len(embedding_samples)):
    for j in range(len(embedding_samples)):
        similarity_matrix[i][j] = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Text {i+1}" for i in range(len(embedding_samples))],
    columns=[f"Text {i+1}" for i in range(len(embedding_samples))]
)

similarity_df.round(3)

,Text 1,Text 2,Text 3,Text 4
Text 1,1.000,0.670,0.076,0.359
Text 2,0.670,1.000,0.110,0.447
Text 3,0.076,0.110,1.000,0.051
Text 4,0.359,0.447,0.051,1.000


## Reasoning scenario setup

**What this demonstrates:** Create a simple policy case that requires applying a rule to a specific member situation.

**What to observe:** The facts are deliberately sufficient so we can compare direct and structured reasoning.


In [25]:
reasoning_case = """
A member has already completed 8 physical therapy visits.
The health plan policy allows 10 visits without prior authorization.
Prior authorization is required starting from the 11th visit.

The provider is requesting the member's 9th physical therapy visit.
"""

reasoning_question = """
Does this visit require prior authorization?
Give only the final decision and one-line justification.
"""

print(reasoning_case)
print(reasoning_question)


A member has already completed 8 physical therapy visits.
The health plan policy allows 10 visits without prior authorization.
Prior authorization is required starting from the 11th visit.

The provider is requesting the member's 9th physical therapy visit.


Does this visit require prior authorization?
Give only the final decision and one-line justification.



**Takeaway:** LLM value appears when it moves from recalling a definition to applying a policy to a case.


## Direct reasoning baseline

**What this demonstrates:** Ask the model to make a decision with minimal reasoning structure.

**What to observe:** The model should reach the correct answer, but the explanation is relatively compressed.


In [27]:
direct_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{reasoning_case}

{reasoning_question}
"""
        }
    ]
)

print(direct_reasoning_response.choices[0].message.content)

No, prior authorization is not required because the 9th visit is within the allowed 10 visits without prior authorization.


**Takeaway:** A correct answer is useful; a reviewable decision path is enterprise-grade.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [28]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

1. Policy rule: The policy allows 10 physical therapy visits without prior authorization; prior authorization is required starting from the 11th visit.  
2. Case fact: The member has completed 8 visits and is requesting the 9th visit.  
3. Decision: The 9th visit does not require prior authorization.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [29]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

1. Policy rule: Prior authorization is required starting from the 11th physical therapy visit; visits 1 through 10 do not require prior authorization.  
2. Case fact: The member has completed 8 visits and is requesting the 9th visit.  
3. Decision: The 9th physical therapy visit does not require prior authorization.


## Ambiguity setup

**What this demonstrates:** Introduce deliberately incomplete policy wording to test how the model behaves when evidence is insufficient.

**What to observe:** The case does not contain enough information for a definitive approval decision.


In [31]:
ambiguous_case = """
A member has completed 10 physical therapy visits.

The policy states:
"Prior authorization may be required after the initial covered visits,
depending on the member's plan and clinical review requirements."

The provider is requesting the 11th visit.
"""

ambiguous_question = """
Does the 11th visit require prior authorization?
"""

print(ambiguous_case)
print(ambiguous_question)


A member has completed 10 physical therapy visits.

The policy states:
"Prior authorization may be required after the initial covered visits,
depending on the member's plan and clinical review requirements."

The provider is requesting the 11th visit.


Does the 11th visit require prior authorization?



**Takeaway:** The most dangerous model failure is often not a wrong answer — it is a confident answer when the evidence is incomplete.


## Overconfidence under ambiguity

**What this demonstrates:** Ask the model for a clear decision even though the policy is conditional.

**What to observe:** Watch for the model converting 'may be required' into a stronger conclusion than the evidence supports.


In [33]:
ambiguous_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

{ambiguous_question}
Give a clear decision and a short justification.
"""
        }
    ]
)

print(ambiguous_response.choices[0].message.content)

**Decision:** Yes, the 11th visit requires prior authorization.

**Justification:** The policy states that prior authorization **may** be required after the initial covered visits depending on the member's plan and clinical review. Since the member has completed the initial 10 visits, any additional visits, including the 11th, would typically require prior authorization to ensure coverage.


**Takeaway:** Fluent language can hide evidentiary gaps. Confidence in wording is not confidence in facts.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [34]:
guardrailed_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only from the provided policy.
2. Do not infer missing policy conditions.
3. If the available information is insufficient for a definitive decision,
   clearly state "Insufficient information".
4. Explain what additional information is required.
"""
        },
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

Question:
Does the 11th visit require prior authorization?
"""
        }
    ]
)

print(guardrailed_response.choices[0].message.content)

The policy states that prior authorization *may* be required after the initial covered visits, depending on the member's plan and clinical review requirements. 

Insufficient information is available to determine if prior authorization is required for the 11th visit in this specific case. 

Additional information needed:
- The details of the member's specific plan regarding physical therapy visits and prior authorization.
- Any clinical review requirements for continuing physical therapy beyond the initial covered visits.


## Overconfidence under ambiguity

**What this demonstrates:** Ask the model for a clear decision even though the policy is conditional.

**What to observe:** Watch for the model converting 'may be required' into a stronger conclusion than the evidence supports.


In [35]:
print("WITHOUT GUARDRAIL")
print("-" * 50)
print(ambiguous_response.choices[0].message.content)

print("\nWITH GUARDRAIL")
print("-" * 50)
print(guardrailed_response.choices[0].message.content)

WITHOUT GUARDRAIL
--------------------------------------------------
**Decision:** Yes, the 11th visit requires prior authorization.

**Justification:** The policy states that prior authorization **may** be required after the initial covered visits depending on the member's plan and clinical review. Since the member has completed the initial 10 visits, any additional visits, including the 11th, would typically require prior authorization to ensure coverage.

WITH GUARDRAIL
--------------------------------------------------
The policy states that prior authorization *may* be required after the initial covered visits, depending on the member's plan and clinical review requirements. 

Insufficient information is available to determine if prior authorization is required for the 11th visit in this specific case. 

Additional information needed:
- The details of the member's specific plan regarding physical therapy visits and prior authorization.
- Any clinical review requirements for contin

**Takeaway:** Fluent language can hide evidentiary gaps. Confidence in wording is not confidence in facts.


## Hallucination test setup

**What this demonstrates:** Ask about a fictional health plan that has never been supplied to the model.

**What to observe:** There is no evidence available from which a factual benefit limit can be determined.


In [36]:
hallucination_question = """
According to the ZS Platinum Plus Health Plan 2026,
what is the maximum number of chiropractic visits allowed per year?
"""

print(hallucination_question)


According to the ZS Platinum Plus Health Plan 2026,
what is the maximum number of chiropractic visits allowed per year?



**Takeaway:** This creates the perfect test of whether the model admits uncertainty or fabricates a plausible policy.


## Hallucination demonstration

**What this demonstrates:** Ask the fictional-plan question without any evidence guardrail.

**What to observe:** If the model invents a specific visit limit, the response is fluent but unsupported.


In [37]:
hallucination_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_response.choices[0].message.content)

According to the ZS Platinum Plus Health Plan 2026, the maximum number of chiropractic visits allowed per year is **20 visits**.


**Takeaway:** The most convincing hallucinations often sound exactly like real policy language.


## Evidence-only guardrail

**What this demonstrates:** Require supporting policy context before the model is allowed to answer.

**What to observe:** The safe response should explicitly state that the answer cannot be determined from available information.


In [38]:
hallucination_guardrail_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only when the required information is present in the provided context.
2. Never invent plan benefits, limits, policies, or coverage rules.
3. If no supporting policy context is provided, respond:
   "I cannot determine this from the available information."
"""
        },
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_guardrail_response.choices[0].message.content)

I cannot determine this from the available information.


**Takeaway:** Grounding is what converts a language model from a plausible storyteller into a trustworthy enterprise assistant.


## Evidence-only guardrail

**What this demonstrates:** Require supporting policy context before the model is allowed to answer.

**What to observe:** The safe response should explicitly state that the answer cannot be determined from available information.


In [39]:
print("WITHOUT EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_response.choices[0].message.content)

print("\nWITH EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_guardrail_response.choices[0].message.content)

WITHOUT EVIDENCE GUARDRAIL
--------------------------------------------------
According to the ZS Platinum Plus Health Plan 2026, the maximum number of chiropractic visits allowed per year is **20 visits**.

WITH EVIDENCE GUARDRAIL
--------------------------------------------------
I cannot determine this from the available information.


**Takeaway:** Grounding is what converts a language model from a plausible storyteller into a trustworthy enterprise assistant.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [40]:
load_dotenv("../.env", override=True)

model_primary = os.getenv("AZURE_OPENAI_MODEL")
model_secondary = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")

print("Primary model   :", model_primary)
print("Secondary model :", model_secondary)

Primary model   : gpt-4.1-mini
Secondary model : gpt-5-mini


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [41]:
comparison_prompt = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Answer using exactly this format:

Decision:
Reason:
"""

## Model suitability comparison

**What this demonstrates:** Run the same healthcare decision prompt against two deployed models and compare behaviour.

**What to observe:** Look beyond fluency: compare evidence discipline, instruction following, latency, and token usage.


In [42]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

PRIMARY MODEL: gpt-4.1-mini
------------------------------------------------------------
Decision: Approved  
Reason: Prior authorization is required starting from the 11th visit, and this is the member's 11th visit request, so the request meets the policy criteria for approval.

SECONDARY MODEL: gpt-5-mini
------------------------------------------------------------
Decision: Denied
Reason: Prior authorization is required beginning with the 11th physical therapy visit; this request is for the 11th visit and the required prior authorization has not been obtained.


**Takeaway:** The best model is not the biggest model — it is the model that meets the workload's quality, latency, and cost requirements.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [43]:
comparison_prompt_guarded = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Rules:
1. Use only the information provided.
2. Do not assume whether authorization has already been requested, approved, or denied.
3. If the information is insufficient for an approval/denial decision, state that clearly.

Answer using exactly this format:

Prior Authorization Required:
Approval Decision:
Reason:
"""

## Model suitability comparison

**What this demonstrates:** Run the same healthcare decision prompt against two deployed models and compare behaviour.

**What to observe:** Look beyond fluency: compare evidence discipline, instruction following, latency, and token usage.


In [44]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

PRIMARY MODEL: gpt-4.1-mini
------------------------------------------------------------
Decision: Approved  
Reason: The request is for the 11th physical therapy visit, which requires prior authorization according to policy. Since this is the initial request for that visit, it can be approved pending authorization.

SECONDARY MODEL: gpt-5-mini
------------------------------------------------------------
Decision: Denied — Prior authorization is required for the 11th physical therapy visit and has not been obtained.
Reason: The member has completed 10 visits; policy requires prior authorization starting with the 11th visit.


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [45]:
def show_metrics(name, response):
    print(name)
    print("-" * 50)

    print("Actual model       :", response.model)
    print("Prompt tokens      :", response.usage.prompt_tokens)
    print("Completion tokens  :", response.usage.completion_tokens)
    print("Total tokens       :", response.usage.total_tokens)

    latency = getattr(response.usage, "latency_checkpoint", None)

    if latency:
        print("Total latency (ms) :", latency.get("total_duration_ms"))
        print("First token (ms)   :", latency.get("user_visible_ttft_ms"))

    print()


show_metrics("GPT-4.1-mini", primary_response)
show_metrics("GPT-5-mini", secondary_response)

GPT-4.1-mini
--------------------------------------------------
Actual model       : gpt-4.1-mini-2025-04-14
Prompt tokens      : 59
Completion tokens  : 44
Total tokens       : 103
Total latency (ms) : 745
First token (ms)   : 221

GPT-5-mini
--------------------------------------------------
Actual model       : gpt-5-mini-2025-08-07
Prompt tokens      : 58
Completion tokens  : 247
Total tokens       : 305
Total latency (ms) : 2212
First token (ms)   : 280



**Takeaway:** Every extra instruction, document chunk, or generated paragraph has a measurable cost footprint — AI architecture is also token architecture.


## Hidden reasoning-token inspection

**What this demonstrates:** Inspect completion-token details to separate visible output from hidden reasoning tokens.

**What to observe:** Reasoning-capable models may consume substantial internal tokens even when the visible answer is short.


In [46]:
print("GPT-4.1-mini token details")
print(primary_response.usage.completion_tokens_details)

print("\nGPT-5-mini token details")
print(secondary_response.usage.completion_tokens_details)

GPT-4.1-mini token details
CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0)

GPT-5-mini token details
CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=192, rejected_prediction_tokens=0)


**Takeaway:** A concise answer can still be an expensive answer — hidden reasoning changes the economics of model selection.


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [47]:
model_comparison = pd.DataFrame([
    {
        "Model": "GPT-4.1-mini",
        "Prompt Tokens": primary_response.usage.prompt_tokens,
        "Completion Tokens": primary_response.usage.completion_tokens,
        "Reasoning Tokens": primary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": primary_response.usage.total_tokens,
        "Latency (ms)": primary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Fast, but made unsupported approval inference"
    },
    {
        "Model": "GPT-5-mini",
        "Prompt Tokens": secondary_response.usage.prompt_tokens,
        "Completion Tokens": secondary_response.usage.completion_tokens,
        "Reasoning Tokens": secondary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": secondary_response.usage.total_tokens,
        "Latency (ms)": secondary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Better evidence discipline, but higher reasoning cost"
    }
])

model_comparison

,Model,Prompt Tokens,Completion Tokens,Reasoning Tokens,Total Tokens,Latency (ms),Observed Behavior
0,GPT-4.1-mini,59,44,0,103,745,"Fast, but made unsupported approval inference"
1,GPT-5-mini,58,247,192,305,2212,"Better evidence discipline, but higher reasoni..."


**Takeaway:** Every extra instruction, document chunk, or generated paragraph has a measurable cost footprint — AI architecture is also token architecture.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [48]:
weak_prompt = """
Explain prior authorization.
"""

weak_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": weak_prompt
        }
    ]
)

print(weak_prompt_response.choices[0].message.content)
print("\nTotal tokens:", weak_prompt_response.usage.total_tokens)

Prior authorization is a requirement from a health insurance company that a healthcare provider get approval before prescribing a specific medication, treatment, or procedure. This process ensures that the prescribed service is medically necessary and appropriate according to the insurer’s coverage policies. 

Typically, the provider submits a request to the insurer, including relevant medical information, and the insurer reviews the request to decide whether to approve or deny it. If approved, the service will be covered under the patient’s plan; if denied, the patient may have to pay out-of-pocket or appeal the decision.

Prior authorization helps manage costs, prevent unnecessary treatments, and promote safe and effective care. However, it can sometimes delay access to care due to the time needed for approval.

Total tokens: 157


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [49]:
strong_prompt = """
You are a healthcare payer domain assistant.

Explain prior authorization to a healthcare technology professional.

Requirements:
1. Use payer terminology.
2. Explain the purpose and workflow.
3. Keep the answer to exactly 3 bullet points.
4. Maximum 80 words.
5. Do not add information beyond the requested scope.
"""

strong_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": strong_prompt
        }
    ]
)

print(strong_prompt_response.choices[0].message.content)
print("\nTotal tokens:", strong_prompt_response.usage.total_tokens)

- Prior authorization (PA) is a payer-driven utilization management process requiring provider submission of clinical documentation before certain services or medications are approved for coverage.  
- Its purpose is to ensure medical necessity, control costs, and enforce benefit policies by validating that the requested service meets evidence-based criteria.  
- Workflow involves provider submitting a PA request via portal or EDI, payer clinical review against criteria, followed by approval, denial, or request for additional info communicated back to the provider.

Total tokens: 167


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [47]:
print("WEAK PROMPT")
print("Prompt tokens     :", weak_prompt_response.usage.prompt_tokens)
print("Completion tokens :", weak_prompt_response.usage.completion_tokens)
print("Total tokens      :", weak_prompt_response.usage.total_tokens)

print("\nSTRONG PROMPT")
print("Prompt tokens     :", strong_prompt_response.usage.prompt_tokens)
print("Completion tokens :", strong_prompt_response.usage.completion_tokens)
print("Total tokens      :", strong_prompt_response.usage.total_tokens)

WEAK PROMPT
Prompt tokens     : 12
Completion tokens : 138
Total tokens      : 150

STRONG PROMPT
Prompt tokens     : 71
Completion tokens : 88
Total tokens      : 159


**Takeaway:** Every extra instruction, document chunk, or generated paragraph has a measurable cost footprint — AI architecture is also token architecture.


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [50]:
prompt_comparison = pd.DataFrame([
    {
        "Prompt Type": "Weak",
        "Prompt Tokens": weak_prompt_response.usage.prompt_tokens,
        "Completion Tokens": weak_prompt_response.usage.completion_tokens,
        "Total Tokens": weak_prompt_response.usage.total_tokens,
        "Control Level": "Low",
        "Output Structure": "Uncontrolled"
    },
    {
        "Prompt Type": "Strong",
        "Prompt Tokens": strong_prompt_response.usage.prompt_tokens,
        "Completion Tokens": strong_prompt_response.usage.completion_tokens,
        "Total Tokens": strong_prompt_response.usage.total_tokens,
        "Control Level": "High",
        "Output Structure": "Controlled"
    }
])

prompt_comparison


,Prompt Type,Prompt Tokens,Completion Tokens,Total Tokens,Control Level,Output Structure
0,Weak,12,145,157,Low,Uncontrolled
1,Strong,71,96,167,High,Controlled
